# Multi-channel RAG — Warhammer 40,000

Prototype de RAG combinant un livre de règles au format PDF et un format de base de données SQLite pour la production de réponses Gemini, avec une conversation multi-tours, et une évaluation préliminaire.

**Environnement visé : Google Colab.** Pour bien fonctionner, il faut exécuter les cellules du notebook dans le bon ordre. Il est impératif d’insérer la clé API Gemini au format `GEMINI_API_KEY` dans les Secrets de Colab.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import json, re, sqlite3, time
from collections import Counter

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/rag-project")
DATA_DIR = PROJECT_DIR / "data" / "data"

PDF_PATH = DATA_DIR / "Core Rules.pdf"
DB_PATH = DATA_DIR / "wahadb.sqlite"

assert PDF_PATH.exists(), f"PDF introuvable : {PDF_PATH}"
assert DB_PATH.exists(), f"Base SQLite introuvable : {DB_PATH}"

print("Projet :", PROJECT_DIR)
print("PDF :", PDF_PATH.name)
print("Base de données :", DB_PATH.name)
print("Fichiers sources vérifiés.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Projet : /content/drive/MyDrive/rag-project
PDF : Core Rules.pdf
Base de données : wahadb.sqlite
Fichiers sources vérifiés.


## 1. Dépendances et configuration

In [ ]:
%pip install -q "ragas==0.4.3" "langchain-community==0.3.31"

## 2. Parsing du livre de règles PDF
Extraction du texte par page, nettoyage léger, suppression prudente des en-têtes/pieds répétés, puis découpage en chunks avec métadonnées de page.

In [ ]:
import json
import re
from collections import Counter

import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Extraction du texte et nettoyage
pages = []

with pymupdf.open(PDF_PATH) as pdf:
    total_pages = len(pdf)

    for page_number, page in enumerate(pdf, start=1):
        text = page.get_text("text")

        # Supprimer les caractères de contrôle parasites
        text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)

        # Nettoyer les espaces sans supprimer les retours à la ligne
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = text.strip()

        if text:
            pages.append({
                "page": page_number,
                "text": text,
            })

# 2. Repérer les lignes répétées en haut ou en bas des pages
# On ne retire que les lignes courtes, présentes sur plusieurs pages,
# et situées au début ou à la fin du texte extrait.
edge_lines = []

for page in pages:
    lines = [line.strip() for line in page["text"].splitlines() if line.strip()]
    edge_lines.extend(lines[:2])
    edge_lines.extend(lines[-2:])

line_counts = Counter(edge_lines)

repeated_lines = {
    line
    for line, count in line_counts.items()
    if count >= max(5, int(total_pages * 0.15))
    and len(line) <= 100
}

# 3. Nettoyer les en-têtes et pieds de page identifiés
for page in pages:
    lines = page["text"].splitlines()
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if stripped in repeated_lines:
            continue

        cleaned_lines.append(line)

    page["text"] = "\n".join(cleaned_lines).strip()

# 4. Découper le texte en chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["text"])

    for chunk_number, chunk_text in enumerate(page_chunks, start=1):
        if chunk_text.strip():
            chunks.append({
                "id": len(chunks),
                "page": page["page"],
                "chunk": chunk_number,
                "text": chunk_text.strip(),
            })

# 5. Enregistrer le résultat
OUTPUT_PATH = PROJECT_DIR / "parsed_rules.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

# 6. Bilan de l'extraction
empty_pages = [
    page_number
    for page_number in range(1, total_pages + 1)
    if page_number not in {page["page"] for page in pages}
]

print(f"Pages dans le PDF : {total_pages}")
print(f"Pages contenant du texte : {len(pages)}")
print(f"Pages sans texte extrait : {empty_pages}")
print(f"Lignes candidates aux en-têtes/pieds de page : {len(repeated_lines)}")
print(f"Nombre de chunks : {len(chunks)}")
print(f"Longueur maximale d'un chunk : {max(len(c['text']) for c in chunks)} caractères")
print(f"Fichier enregistré : {OUTPUT_PATH}")

Pages dans le PDF : 60
Pages contenant du texte : 59
Pages sans texte extrait : [2]
Lignes candidates aux en-têtes/pieds de page : 2
Nombre de chunks : 191
Longueur maximale d'un chunk : 998 caractères
Fichier enregistré : /content/drive/MyDrive/rag-project/parsed_rules.json


In [ ]:
# Vérifications finales du fichier parsé

control_chars = [
    (chunk["id"], chunk["page"])
    for chunk in chunks
    if re.search(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", chunk["text"])
]

missing_metadata = [
    chunk["id"]
    for chunk in chunks
    if not all(key in chunk for key in ("id", "page", "chunk", "text"))
]

print("Chunks avec caractères de contrôle :", len(control_chars))
print("Chunks avec métadonnées manquantes :", len(missing_metadata))
print("Premier chunk :", chunks[0])

Chunks avec caractères de contrôle : 0
Chunks avec métadonnées manquantes : 0
Premier chunk : {'id': 0, 'page': 1, 'chunk': 1, 'text': 'Core Rules\n‘We are beset on all sides by vile predatory aliens \nand sedition gnaws at us from within; in this dark hour \nthe best we can do is look to our wargear and pray to \nour gods.’\n- Skolak a’Trellar IV, Imperial Commander'}


## 3. Recherche sémantique dans le PDF

In [ ]:
import json
import torch
from sentence_transformers import SentenceTransformer, util

# Charger les chunks parsés
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

# Charger le modèle d'embeddings
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# Calculer les embeddings des chunks
texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

def search_rules(question, top_k=3):
    """Retrouver les passages les plus pertinents pour une question."""

    question_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    scores = util.cos_sim(
        question_embedding,
        chunk_embeddings,
    )[0]

    top_results = torch.topk(
        scores,
        k=min(top_k, len(chunks)),
    )

    results = []

    for score, index in zip(top_results.values, top_results.indices):
        chunk = chunks[index.item()]

        results.append({
            "page": chunk["page"],
            "text": chunk["text"],
            "score": round(score.item(), 4),
        })

    return results

print(f"Moteur de recherche prêt : {len(chunks)} chunks indexés.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Moteur de recherche prêt : 191 chunks indexés.


## 4. Recherche dans SQLite

In [ ]:
def get_unit_details(unit_name):
    """Récupère les caractéristiques et les coûts disponibles d'une unité."""
    with sqlite3.connect(DB_PATH) as conn:
        return conn.execute(
            """
            SELECT
                d.name, f.name AS faction, m.name AS model_name,
                m.m AS movement, m.t AS toughness, m.sv AS save_value,
                m.w AS wounds, c.description AS cost_description, c.cost AS cost
            FROM datasheets AS d
            LEFT JOIN factions AS f ON d.factionId = f.Id
            LEFT JOIN datasheets_models AS m ON d.Id = m.DatasheetId
            LEFT JOIN datasheets_models_cost AS c ON d.Id = c.DatasheetId
            WHERE d.name = ?
              AND c.description GLOB '[0-9]* models'
              AND c.cost != ''
            """,
            (unit_name,),
        ).fetchall()


## 5. Routage et récupération multicanale
Le routeur actuel est une première version fondée sur des mots-clés. La recherche SQL est limitée aux unités explicitement reconnues ci-dessous.

In [ ]:
def route_question(question):
    """
    Choisit la source à interroger en fonction de la question.
    Première version : routage par mots-clés.
    """
    question_lower = question.lower()

    sql_keywords = [
        "caractéristique", "caractéristiques",
        "statistique", "statistiques",
        "mouvement", "endurance", "sauvegarde",
        "blessures", "points", "coût", "coûts",
        "faction", "équipement", "unité", "unités",
        "datasheet", "custodian", "terminator",
    ]

    if any(keyword in question_lower for keyword in sql_keywords):
        return "sql"

    return "pdf"

In [ ]:
def retrieve_information(question, top_k=3):
    """
    Oriente la question vers la source appropriée
    et récupère les informations correspondantes.
    """
    source = route_question(question)

    if source == "pdf":
        results = search_rules(question, top_k=top_k)

        return {
            "question": question,
            "source": "pdf",
            "results": results,
        }

    # Première version SQL : recherche d'une unité par son nom.
    unit_names = [
        "Custodian Guard",
        "Venerable Contemptor Dreadnought",
        "Venerable Land Raider",
    ]

    question_lower = question.lower()
    matched_unit = next(
        (
            name for name in unit_names
            if name.lower() in question_lower
        ),
        None,
    )

    if matched_unit is None:
        return {
            "question": question,
            "source": "sql",
            "results": [],
            "message": "Aucune unité reconnue dans cette version du prototype.",
        }

    results = get_unit_details(matched_unit)

    return {
        "question": question,
        "source": "sql",
        "results": results,
    }

## 6. Préparation du contexte et génération

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Ajoute GEMINI_API_KEY dans les Secrets de Google Colab.")

client = genai.Client(api_key=api_key)
print("Client Gemini initialisé.")


Client Gemini initialisé.


In [ ]:
def format_context(retrieval_result):
    """Transforme les résultats récupérés en contexte lisible."""

    source = retrieval_result["source"]
    results = retrieval_result["results"]

    if not results:
        return "Aucune information pertinente n'a été récupérée."

    if source == "pdf":
        passages = []

        for result in results:
            passages.append(
                f"[Page {result['page']}]\n{result['text']}"
            )

        return "\n\n".join(passages)

    # Résultats SQL
    formatted_rows = []

    for row in results:
        (
            unit_name,
            faction,
            model_name,
            movement,
            toughness,
            save_value,
            wounds,
            cost_description,
            cost,
        ) = row

        formatted_rows.append(
            f"Unité : {unit_name}\n"
            f"Faction : {faction}\n"
            f"Modèle : {model_name}\n"
            f"Mouvement : {movement}\n"
            f"Endurance : {toughness}\n"
            f"Sauvegarde : {save_value}\n"
            f"Blessures : {wounds}\n"
            f"Configuration : {cost_description}\n"
            f"Coût : {cost} points"
        )

    return "\n\n".join(formatted_rows)

In [ ]:
def generate_answer(question, retrieval_result):
    """Génère une réponse à partir des informations récupérées."""

    context = format_context(retrieval_result)

    prompt = f"""
Tu es un assistant spécialisé dans les règles de Warhammer 40,000.

Réponds à la question de l'utilisateur en utilisant uniquement
les informations présentes dans le contexte fourni.

Consignes :
- Réponds dans la langue de la question.
- Sois clair, précis et concis.
- Pour les informations provenant du PDF, indique les numéros de page
  lorsqu'ils sont disponibles.
- Si le contexte ne permet pas de répondre, indique-le clairement.
- N'invente aucune règle, caractéristique ou valeur.
- Distingue les informations explicitement présentes dans le contexte
  de toute conclusion que tu en tires.

Contexte :
{context}

Question :
{question}

Réponse :
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config={
            "thinking_config": {
                "thinking_budget": 0
            }
        }
    )

    return response.text

## 7. Tests et évaluation de base
Le jeu de test est volontairement petit : il sert de vérification initiale, pas d'évaluation exhaustive.

In [ ]:
questions_evaluation = [
    {
        "question": "What happens when a Transport is destroyed?",
        "expected_source": "pdf",
    },
    {
        "question": "What are the characteristics of the Custodian Guard?",
        "expected_source": "sql",
    },
    {
        "question": "How many points does a Custodian Guard unit cost?",
        "expected_source": "sql",
    },
    {
        "question": "Can a unit charge after disembarking from a destroyed Transport?",
        "expected_source": "pdf",
    },
]

print(f"Nombre de questions préparées : {len(questions_evaluation)}")

for item in questions_evaluation:
    print(f"- [{item['expected_source'].upper()}] {item['question']}")

Nombre de questions préparées : 4
- [PDF] What happens when a Transport is destroyed?
- [SQL] What are the characteristics of the Custodian Guard?
- [SQL] How many points does a Custodian Guard unit cost?
- [PDF] Can a unit charge after disembarking from a destroyed Transport?


In [ ]:
import json
import time
from pathlib import Path

resultats_evaluation = []

for i, item in enumerate(questions_evaluation, start=1):
    question = item["question"]

    print(f"\n[{i}/{len(questions_evaluation)}] {question}")

    try:
        # Récupérer les informations
        retrieval_result = retrieve_information(question)

        # Préparer le contexte
        context = format_context(retrieval_result)

        # Générer la réponse
        answer = generate_answer(question, retrieval_result)

        # Enregistrer les résultats
        resultats_evaluation.append({
            "question": question,
            "expected_source": item["expected_source"],
            "actual_source": retrieval_result["source"],
            "contexts": [context],
            "answer": answer,
        })

        print(f"Source : {retrieval_result['source']}")
        print(f"Réponse : {answer[:400]}")

    except Exception as e:
        print(f"Erreur : {e}")

    # Petite pause entre les appels API
    time.sleep(1)

# Enregistrer les résultats dans un fichier JSON
EVALUATION_PATH = PROJECT_DIR / "evaluation_results.json"

with open(EVALUATION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        resultats_evaluation,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    f"\nÉvaluation terminée : "
    f"{len(resultats_evaluation)}/{len(questions_evaluation)} questions traitées."
)
print(f"Résultats enregistrés dans : {EVALUATION_PATH}")


[1/4] What happens when a Transport is destroyed?
Source : pdf
Réponse : When a Transport model is destroyed, the following rules apply (page 18):

*   **Immediate Disembarkation:** Any units embarked within the Transport must immediately disembark before the model is removed from the battlefield.
*   **Mortal Wounds:** You must roll one D6 for each disembarking model. For each roll of 1, the unit suffers 1 mortal wound.
*   **Battle-shock:** The unit is Battle-shocked

[2/4] What are the characteristics of the Custodian Guard?
Source : sql
Réponse : The characteristics of the Custodian Guard, according to the provided context, are as follows:

*   **Faction:** Adeptus Custodes
*   **Model:** Custodian Guard
*   **Movement:** 6"
*   **Endurance:** 6
*   **Save:** 2+
*   **Wounds:** 3

The context provides four different configurations for this unit:
*   4 models at 170 points.
*   4 models at 180 points.
*   5 models at 215 points.
*   5 model

[3/4] How many points does a Custodian G

In [ ]:
references = [
    (
        "When a Transport model is destroyed, embarked units must "
        "immediately disembark before the Transport is removed. "
        "Roll one D6 for each disembarking model; each roll of 1 "
        "causes 1 mortal wound to its unit. The unit becomes "
        "Battle-shocked until the start of its controlling player's "
        "next Command phase, counts as having made a Normal move, "
        "and cannot declare a charge until the end of the turn. "
        "It is not affected by the Transport's Deadly Demise ability."
    ),
    (
        "The Custodian Guard belong to the Adeptus Custodes faction. "
        "Their characteristics are Movement 6 inches, Toughness 6, "
        "Save 2+, and 3 Wounds. The listed costs are 170 or 180 points "
        "for 4 models, and 215 or 225 points for 5 models."
    ),
    (
        "A Custodian Guard unit costs 170 or 180 points for 4 models, "
        "and 215 or 225 points for 5 models. The available data "
        "contains multiple cost categories."
    ),
    (
        "No. A unit that disembarks from a destroyed Transport counts "
        "as having made a Normal move and cannot declare a charge "
        "until the end of the turn."
    ),
]

assert len(references) == len(resultats_evaluation)

for i, reference in enumerate(references, start=1):
    resultats_evaluation[i - 1]["reference"] = reference

with open(EVALUATION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        resultats_evaluation,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"{len(references)} réponses de référence ajoutées.")
print(f"Résultats mis à jour : {EVALUATION_PATH}")

4 réponses de référence ajoutées.
Résultats mis à jour : /content/drive/MyDrive/rag-project/evaluation_results.json


## 8. Évaluation RAGAS — Faithfulness
Cette section utilise l'adaptateur Instructor avec le client Google Gen AI. Les dépendances RAGAS peuvent être sensibles aux versions de l'environnement Colab.

In [ ]:
from ragas.llms.base import InstructorLLM
from ragas.metrics.collections import Faithfulness

print("Imports RAGAS réussis !")

Imports RAGAS réussis !


In [ ]:
%pip install -q langchain-community

import instructor
from ragas.llms.base import InstructorLLM
from ragas.metrics.collections import Faithfulness

async_instructor_client = instructor.from_genai(client=client, use_async=True)
ragas_llm = InstructorLLM(
    client=async_instructor_client,
    model="gemini-3.1-flash-lite",
    provider="google",
)
faithfulness_metric = Faithfulness(llm=ragas_llm)
print("Métrique RAGAS initialisée :", faithfulness_metric.name)

Métrique RAGAS initialisée : faithfulness


In [ ]:
import pandas as pd

resultats_faithfulness = []

for i, item in enumerate(resultats_evaluation, start=1):
    try:
        score = await faithfulness_metric.ascore(
            user_input=item["question"],
            response=item["answer"],
            retrieved_contexts=item["contexts"],
        )

        resultats_faithfulness.append({
            "question": item["question"],
            "expected_source": item["expected_source"],
            "actual_source": item["actual_source"],
            "faithfulness": score.value,
        })

        print(f"Question {i}/{len(resultats_evaluation)} : {score.value}")

    except Exception as e:
        print(f"Erreur pour la question {i} : {type(e).__name__}: {e}")

df_faithfulness = pd.DataFrame(resultats_faithfulness)

display(df_faithfulness)

if not df_faithfulness.empty:
    print(
        "Faithfulness moyen :",
        round(df_faithfulness["faithfulness"].mean(), 3)
    )

Question 1/4 : 1.0
Question 2/4 : 1.0


ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: 'NoneType' object is not iterable


Erreur pour la question 3 : InstructorRetryException: 'NoneType' object is not iterable
Question 4/4 : 1.0


,question,expected_source,actual_source,faithfulness
0,What happens when a Transport is destroyed?,pdf,pdf,1.0
1,What are the characteristics of the Custodian ...,sql,sql,1.0
2,Can a unit charge after disembarking from a de...,pdf,pdf,1.0


Faithfulness moyen : 1.0


In [ ]:
test_controle = await faithfulness_metric.ascore(
    user_input="Can a unit charge after disembarking from a destroyed Transport?",
    response=(
        "Yes. A unit that disembarks from a destroyed Transport "
        "can declare a charge during the same turn."
    ),
    retrieved_contexts=[
        """[Page 18]
        Until the end of the turn, that unit counts as having made
        a Normal move this turn, and cannot declare a charge this turn."""
    ],
)

print("Score du test de contrôle :", test_controle.value)

Score du test de contrôle : 0.0


## 9. Chatbot multi-tours
L'historique est utilisé pour reformuler les questions de suivi avant la recherche documentaire, puis pour contextualiser la réponse.

In [ ]:
conversation_history = []

def chat(question):
    """Répond à une question en tenant compte de l'historique."""

    # 1. Reformuler la question pour rendre explicites les références
    history = ""
    for message in conversation_history:
        history += f"{message['role']}: {message['content']}\n"

    if history:
        rewrite_prompt = f"""
Tu aides un système RAG à rechercher des règles de Warhammer 40,000.

À partir de l'historique, reformule la question actuelle en une question
autonome, compréhensible sans l'historique.

- Conserve exactement l'intention de l'utilisateur.
- Résous les pronoms et les références aux éléments précédemment évoqués.
- Ne réponds pas à la question.
- N'ajoute aucune règle ou information.
- Retourne uniquement la question reformulée.

Historique :
{history}

Question actuelle :
{question}
"""
        rewrite_response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=rewrite_prompt,
            config={"thinking_config": {"thinking_budget": 0}},
        )
        search_question = rewrite_response.text.strip()
    else:
        search_question = question

    # 2. Rechercher les informations avec la question reformulée
    retrieval_result = retrieve_information(search_question)
    context = format_context(retrieval_result)

    # 3. Générer la réponse en tenant compte de l'historique
    prompt = f"""
Tu es un assistant spécialisé dans les règles de Warhammer 40,000.

Réponds à la question actuelle en utilisant le contexte récupéré
et l'historique de conversation.

Règles :
- Réponds dans la langue de l'utilisateur.
- Utilise uniquement les informations du contexte pour les règles du jeu.
- N'invente aucune règle.
- Si le contexte ne permet pas de répondre, dis-le clairement.

Historique de conversation :
{history}

Question originale :
{question}

Question utilisée pour la recherche :
{search_question}

Contexte récupéré :
{context}
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config={"thinking_config": {"thinking_budget": 0}},
    )

    answer = response.text

    conversation_history.append({
        "role": "user",
        "content": question,
    })
    conversation_history.append({
        "role": "assistant",
        "content": answer,
    })

    return answer

In [ ]:
conversation_history.clear()

print("QUESTION 1")
print("What happens when a Transport is destroyed?\n")
print(chat("What happens when a Transport is destroyed?"))

print("\n" + "=" * 60 + "\n")

print("QUESTION 2")
print("Can they charge afterward?\n")
print(chat("Can they charge afterward?"))

QUESTION 1
What happens when a Transport is destroyed?

Lorsqu'un modèle de transport est détruit, voici ce qui se passe selon les règles :

*   **Débarquement immédiat :** Toutes les unités embarquées doivent immédiatement débarquer avant que le modèle de transport ne soit retiré du champ de bataille.
*   **Dommages subis :** Vous devez lancer un D6 pour chaque figurine qui débarque. Pour chaque résultat de 1, l'unité de cette figurine subit 1 blessure mortelle. Les unités ne sont pas affectées par l'aptitude "Deadly Demise" du transport.
*   **État de l'unité :**
    *   L'unité est considérée comme "Battle-shocked" jusqu'au début de la prochaine phase de Commandement de son joueur.
    *   Jusqu'à la fin du tour, l'unité compte comme ayant effectué un mouvement Normal et ne peut pas déclarer de charge.

**Débarquement d'urgence (Emergency Disembarkation) :**
Si, lors de la destruction, il n'est pas possible de placer une unité entièrement à moins de 3" du transport et hors de portée

## Limites et pistes d'amélioration

Le prototype est fonctionnel, mais plusieurs aspects pourraient être largements améliorés :

- **Routage PDF/SQL :** le routage est basé sur des mots-clés. Il faudrait à élargir la couverture des formulations et à mieux gérer les questions nécessitant les deux sources.
- **Requêtes SQL :** les unités interrogeables sont limitées à une liste explicite. La reconnaissance des unités et la gestion de leurs différentes configurations doivent être généralisées.
- **Gestion des coûts :** certaines unités possèdent plusieurs configurations et plusieurs coûts. Le système devrait mieux gérer ces ambiguïtés avant de fournir une réponse unique.
- **Évaluation :** les tests se limitent à quatre questions et à la métrique *Faithfulness*. Un jeu de données plus large permettrait de mieux éavluer la récupération et la génération. La configuration de RAGAS reste par ailleurs à stabiliser.
- **Parsing PDF :** une page sans texte extractible n'est pas prise en compte. Un traitement complémentaire, par exemple par OCR, pourrait régler ce problème.
- **Chat multi-tours :** le fonctionnement a été vérifié sur un seul exemple, mais des tests supplémentaires seraient nécessaires pour évaluer sa robustesse dans des conversations plus variées.

Ces pistes n'ont pas pu être explorées en profondeur dans le temps imparti.